# 01 — Data Understanding

**Business problem.** Banks approve/decline card transactions in milliseconds. Fraud is ~0.17% of
transactions but each miss costs the full amount plus chargeback fees, while each false alarm costs
customer friction and support time. We need a model that outputs a **fraud probability**, decided by a
**business-optimized threshold**.

**Dataset.** Kaggle *Credit Card Fraud Detection*: 284,807 transactions over 2 days by European
cardholders, 492 frauds (0.172%). `V1–V28` are PCA-anonymized features; only `Time` (seconds since
first transaction), `Amount`, and the target `Class` are raw.

All reusable logic lives in `src/`; this notebook only orchestrates.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")
%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import Image, display
from src.utils import load_config, resolve_path

config = load_config()
pd.set_option("display.max_columns", 40)

In [2]:
from src.data_loader import download_data, load_raw_data

download_data(config)          # no-op if data/raw/creditcard.csv already exists
df = load_raw_data(config)
df.head()

2026-07-31 18:14:53 | INFO    | src.data_loader | Dataset already present at /Users/mohitpatle/Library/CloudStorage/OneDrive-RelianceFoundationInstitutionofEducationandResearch/Credit card fraud detection/data/raw/creditcard.csv (150.8 MB)


2026-07-31 18:14:54 | INFO    | src.data_loader | Loaded raw data: 284807 rows x 31 columns


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,0.090794,-0.551600,-0.617801,-0.991390,-0.311169,1.468177,-0.470401,0.207971,0.025791,0.403993,0.251412,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,-0.166974,1.612727,1.065235,0.489095,-0.143772,0.635558,0.463917,-0.114805,-0.183361,-0.145783,-0.069083,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,0.207643,0.624501,0.066084,0.717293,-0.165946,2.345865,-2.890083,1.109969,-0.121359,-2.261857,0.524980,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,-0.054952,-0.226487,0.178228,0.507757,-0.287924,-0.631418,-1.059647,-0.684093,1.965775,-1.232622,-0.208038,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,0.753074,-0.822843,0.538196,1.345852,-1.119670,0.175121,-0.451449,-0.237033,-0.038195,0.803487,0.408542,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


## Structure and types

31 columns: `Time`, `V1–V28`, `Amount`, `Class`. Everything is numeric — no categorical encoding
needed. `V1–V28` are already the output of a PCA transformation done by the dataset publishers for
confidentiality, so they are centered, uncorrelated with each other, and unitless.

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 21  V21     284807 non-nu

In [4]:
df.describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
Time,284807.0,94813.860,47488.146,0.000,54201.500,84692.000,139320.500,172792.000
V1,284807.0,0.000,1.959,-56.408,-0.920,0.018,1.316,2.455
V2,284807.0,0.000,1.651,-72.716,-0.599,0.065,0.804,22.058
V3,284807.0,-0.000,1.516,-48.326,-0.890,0.180,1.027,9.383
V4,284807.0,0.000,1.416,-5.683,-0.849,-0.020,0.743,16.875
V5,284807.0,0.000,1.380,-113.743,-0.692,-0.054,0.612,34.802
V6,284807.0,0.000,1.332,-26.161,-0.768,-0.274,0.399,73.302
V7,284807.0,-0.000,1.237,-43.557,-0.554,0.040,0.570,120.589
V8,284807.0,0.000,1.194,-73.217,-0.209,0.022,0.327,20.007
V9,284807.0,-0.000,1.099,-13.434,-0.643,-0.051,0.597,15.595


## Data validation

Automated checks (schema, missing values, duplicates, invalid values, target distribution) run via
`src.data_validation`. The report is persisted to `artifacts/validation_report.json` so every run
leaves an audit trail.

Key expected findings:
- **No missing values** anywhere (rare luxury — the dataset is pre-cleaned).
- **1,081 duplicate rows** — dropped *before* splitting so identical rows can never land in both
  train and test (that would leak).
- **Extreme imbalance**: 492 frauds / 284,807 rows ≈ 0.172% → accuracy is meaningless.

In [5]:
from src.data_validation import run_validation

report = run_validation(df, config)
report

2026-07-31 18:14:55 | INFO    | src.data_validation | Validation [schema]: PASS


2026-07-31 18:14:55 | INFO    | src.data_validation | Validation [missing]: PASS


2026-07-31 18:14:55 | INFO    | src.data_validation | Validation [duplicates]: ATTENTION {'passed': False, 'n_duplicates': 1081}


2026-07-31 18:14:55 | INFO    | src.data_validation | Validation [invalid_values]: PASS


2026-07-31 18:14:55 | INFO    | src.data_validation | Validation [target_distribution]: PASS


2026-07-31 18:14:55 | INFO    | src.data_validation | Validation report saved to artifacts/validation_report.json


{'n_rows': 284807,
 'n_columns': 31,
 'schema': {'passed': True, 'issues': []},
 'missing': {'passed': True, 'missing_by_column': {}, 'total_missing': 0},
 'duplicates': {'passed': False, 'n_duplicates': 1081},
 'invalid_values': {'passed': True, 'issues': []},
 'target_distribution': {'passed': True,
  'n_genuine': 284315,
  'n_fraud': 492,
  'fraud_rate': 0.001727,
  'imbalance_ratio': 577.9},
 'passed': True}